<!-- course-header -->
# Lesson 27: Computer vision: detection, segmentation and beyond

*Companion notebook for lesson 27. Read the lesson [online](https://ashutoshjoshi1.github.io/AI-ZerotoMastery/lessons/27-computer-vision.html) (it works well on a phone) or [offline](../lessons/27-computer-vision.html). Worked solutions are in `solutions/27-computer-vision-solutions.ipynb`; try each exercise first.*

<a href="https://colab.research.google.com/github/ashutoshjoshi1/AI-ZerotoMastery/blob/main/notebooks/27-computer-vision.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

Go beyond labelling whole images: implement IoU, non-maximum suppression and mAP yourself, train a small object detector and a U-Net segmenter on a CPU, then put pretrained, promptable and open-vocabulary models to work, track objects through video and get vision models ready to deploy.

**Runs on:** Any computer (CPU), offline. The detector and the U-Net each train in about a minute on a laptop CPU; one optional cell downloads a 74 MB pretrained detector.

**How to use it:** read a section of the lesson, then run the matching cells here, in order. Each exercise has a cell titled *Hands-on* with the same name as in the lesson. Change numbers, break things on purpose, and note what happens in your learning log. Cells that need an API key, a GPU or a download say so in their first line.

## Setup

Install what this lesson needs. Skip this cell if your environment from lesson 00 already has these packages.

In [ ]:
%pip install -q numpy matplotlib torch torchvision

## Beyond "what is in this image"

Classification gives one label per image. Detection, segmentation, keypoints, depth, tracking and OCR ask where, which pixels, how far and which object is which. The lesson page has a table of tasks, typical models and metrics.

## Boxes and how to compare them

### Box formats

torchvision and Pascal VOC use corners (xyxy), COCO uses top-left plus size (xywh), and YOLO uses the centre plus size, normalised by the image size. Convert once, where data enters your code.

In [ ]:
import numpy as np
import torch
from torchvision.ops import box_convert

def xywh_to_xyxy(b):
    b = np.asarray(b, dtype=float)
    return np.concatenate([b[..., :2], b[..., :2] + b[..., 2:]], axis=-1)

def xyxy_to_cxcywh(b):
    b = np.asarray(b, dtype=float)
    wh = b[..., 2:] - b[..., :2]
    return np.concatenate([b[..., :2] + wh / 2, wh], axis=-1)

W, H = 64, 48                                  # image width and height
coco_box = np.array([[10, 20, 30, 16]])        # COCO style: x, y, width, height
box = xywh_to_xyxy(coco_box)                   # torchvision style: x1, y1, x2, y2
yolo = xyxy_to_cxcywh(box) / [W, H, W, H]      # YOLO style: normalised centre and size
print(box, yolo.round(3))
print(box_convert(torch.tensor(coco_box), "xywh", "xyxy"))

### Intersection over union

IoU is the shared area divided by the total covered area. The overlap of two boxes is a box: the larger top-left corner and the smaller bottom-right corner, clipped at zero.

In [ ]:
from torchvision.ops import box_iou

def box_area(b):
    return (b[..., 2] - b[..., 0]) * (b[..., 3] - b[..., 1])

def iou_matrix(a, b):
    """Pairwise IoU between boxes a (N, 4) and b (M, 4), both xyxy."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    top_left = np.maximum(a[:, None, :2], b[None, :, :2])      # (N, M, 2)
    bottom_right = np.minimum(a[:, None, 2:], b[None, :, 2:])  # (N, M, 2)
    wh = np.clip(bottom_right - top_left, 0, None)            # 0 if no overlap
    inter = wh[..., 0] * wh[..., 1]
    union = box_area(a)[:, None] + box_area(b)[None, :] - inter
    return inter / union

a = np.array([[0, 0, 10, 10], [5, 5, 15, 15]])
b = np.array([[0, 0, 10, 10], [20, 20, 30, 30], [0, 0, 10, 5]])
print(iou_matrix(a, b).round(3))

rng = np.random.default_rng(0)
xy = rng.uniform(0, 100, (500, 2))
boxes = np.hstack([xy, xy + rng.uniform(5, 40, (500, 2))])
ours = iou_matrix(boxes, boxes)
theirs = box_iou(torch.tensor(boxes), torch.tensor(boxes)).numpy()
print(np.abs(ours - theirs).max())

### Non-maximum suppression

Keep the best box, delete boxes that overlap it by more than the threshold, repeat.

In [ ]:
from torchvision.ops import nms

def nms_numpy(boxes, scores, iou_thresh=0.5):
    order = np.argsort(-scores)                # best first
    keep = []
    while order.size > 0:
        best = order[0]
        keep.append(best)
        rest = order[1:]
        overlap = iou_matrix(boxes[best][None], boxes[rest])[0]
        order = rest[overlap <= iou_thresh]    # drop boxes that overlap the winner
    return np.array(keep)

# 3 objects, 40 jittered candidate boxes around each
centres = np.array([[20, 20, 50, 60], [45, 30, 90, 70], [10, 70, 40, 95]])
cands = np.repeat(centres, 40, axis=0) + rng.normal(0, 3, (120, 4))
scores = rng.uniform(0, 1, 120)
keep = nms_numpy(cands, scores, 0.5)
keep_tv = nms(torch.tensor(cands), torch.tensor(scores), 0.5).numpy()
print(len(keep), np.array_equal(keep, keep_tv))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
for ax, subset, title in [(axes[0], cands, "all 120 candidates"), (axes[1], cands[keep], "after NMS")]:
    for x1, y1, x2, y2 in subset:
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, alpha=0.5))
    ax.set_xlim(0, 100); ax.set_ylim(100, 0); ax.set_title(title)
plt.show()

## How detectors work

### Two-stage: propose, then classify

Faster R-CNN: a region proposal network scores anchor boxes, then RoI pooling crops features for each proposal, which a second head classifies and refines.

### One-stage: predict on a grid

YOLO and SSD predict boxes and classes from every grid cell in one pass. Anchor-free detectors such as FCOS and CenterNet predict boxes from locations or centre heatmaps.

### Set prediction: DETR

Object queries in a transformer decoder each predict one box or "no object". Training matches predictions to objects one-to-one with the Hungarian algorithm, so duplicates are penalised and no NMS is needed. Below, a toy matching by brute force.

In [ ]:
from itertools import permutations

pred_boxes = np.array([[10, 10, 30, 30], [12, 11, 31, 29],
                       [50, 50, 80, 90], [0, 60, 20, 80]])
pred_prob = np.array([[0.7, 0.2, 0.1],     # columns: cat, dog, no object
                      [0.6, 0.3, 0.1],
                      [0.1, 0.8, 0.1],
                      [0.1, 0.1, 0.8]])
gt_boxes = np.array([[11, 10, 30, 30], [52, 48, 80, 88]])
gt_labels = np.array([0, 1])                # a cat and a dog

# cost of assigning prediction i to object j: low when the class is
# probable and the boxes overlap
cost = -pred_prob[:, gt_labels] + (1 - iou_matrix(pred_boxes, gt_boxes))
best = min(permutations(range(len(pred_boxes)), len(gt_boxes)),
           key=lambda p: cost[list(p), range(len(gt_boxes))].sum())
print(cost.round(2))
print("prediction matched to each object:", best)

## Measuring a detector: precision, recall and mAP

Sort detections by score, mark each as a true or false positive at an IoU threshold, and integrate precision over recall. COCO averages over classes and over IoU thresholds from 0.5 to 0.95.

In [ ]:
def average_precision(det_img, det_score, det_box, gt_boxes, iou_thresh=0.5):
    """AP for one class, COCO-style 101-point interpolation.
    det_*: arrays over all detections; gt_boxes: list of (K_i, 4) per image."""
    n_gt = sum(len(g) for g in gt_boxes)
    matched = [np.zeros(len(g), dtype=bool) for g in gt_boxes]
    tp = np.zeros(len(det_score))
    for k in np.argsort(-det_score):           # most confident first
        g = gt_boxes[det_img[k]]
        if len(g) == 0:
            continue
        ious = iou_matrix(det_box[k][None], g)[0]
        ious[matched[det_img[k]]] = -1         # each object can be found once
        j = ious.argmax()
        if ious[j] >= iou_thresh:
            matched[det_img[k]][j] = True
            tp[k] = 1
    tp = tp[np.argsort(-det_score)]
    recall = np.cumsum(tp) / n_gt
    precision = np.cumsum(tp) / np.arange(1, len(tp) + 1)
    # best precision achievable at this recall or higher
    precision = np.maximum.accumulate(precision[::-1])[::-1]
    levels = np.linspace(0, 1, 101)
    idx = np.searchsorted(recall, levels, side="left")
    return np.mean([precision[i] if i < len(precision) else 0 for i in idx])

# three images of one class; five objects in total
gt = [np.array([[10, 10, 40, 40], [50, 50, 90, 90]]),
      np.array([[20, 30, 60, 70]]),
      np.array([[0, 0, 30, 20], [60, 10, 95, 40]])]
det_img = np.array([0, 0, 0, 1, 1, 2, 2])
det_score = np.array([0.95, 0.90, 0.60, 0.85, 0.40, 0.80, 0.30])
det_box = np.array([[11, 10, 41, 40],      # good
                    [50, 52, 88, 90],      # good
                    [12, 12, 40, 41],      # duplicate of the first
                    [24, 34, 62, 72],      # IoU about 0.7
                    [70, 70, 95, 95],      # background
                    [2, 1, 28, 24],        # IoU about 0.7
                    [66, 14, 99, 44]])     # IoU about 0.6
for t in (0.5, 0.75):
    print(f"AP@{t}: {average_precision(det_img, det_score, det_box, gt, t):.3f}")
thresholds = np.arange(0.5, 0.96, 0.05)
aps = [average_precision(det_img, det_score, det_box, gt, t) for t in thresholds]
print(f"AP@[.5:.95]: {np.mean(aps):.3f}")

In [ ]:
import matplotlib.pyplot as plt

plt.plot(thresholds, aps, marker="o")
plt.xlabel("IoU threshold"); plt.ylabel("AP"); plt.title("Stricter matching, lower AP")
plt.show()

### Hands-on: class-aware NMS

Plain NMS lets a confident "person" box delete an overlapping "horse" box, which is wrong when someone is riding the horse.

1. Write `batched_nms_numpy(boxes, scores, labels, iou_thresh)`, in which a box can only suppress boxes of its own class. Hint: you can reuse `nms_numpy` unchanged if you first move each class's boxes so far apart that boxes of different classes can never overlap.
2. Check that it returns exactly the same indices as `torchvision.ops.batched_nms` for 300 random boxes in three classes.
3. Make a person box and a horse box with an IoU above 0.5, and show that plain NMS keeps one of them while yours keeps both.

In [ ]:
# Your code here

## Train a tiny detector

### A synthetic dataset

64x64 images with one to three squares, circles and triangles. Boxes and masks come for free because you draw the shapes.

In [ ]:
CLASSES = ["square", "circle", "triangle"]
SIZE = 64

def make_scene(rng, max_objects=3):
    """One 64x64 RGB image with 1 to 3 shapes, their boxes, labels and mask."""
    yy, xx = np.mgrid[:SIZE, :SIZE]
    bg = rng.uniform(0.2, 0.8, 3)
    tilt = rng.uniform(-0.3, 0.3, 2)
    img = bg + (tilt[0] * (xx / SIZE - 0.5) + tilt[1] * (yy / SIZE - 0.5))[..., None]
    img = img + rng.normal(0, 0.08, (SIZE, SIZE, 3))
    mask = np.zeros((SIZE, SIZE), dtype=np.int64)       # 0 means background
    boxes, labels = [], []
    for _ in range(rng.integers(1, max_objects + 1)):
        for _attempt in range(20):                      # find a free spot
            s = rng.integers(12, 25)
            cx, cy = rng.integers(s // 2, SIZE - s // 2, 2)
            box = np.array([cx - s / 2, cy - s / 2, cx + s / 2, cy + s / 2])
            if not boxes or iou_matrix(box[None], np.array(boxes)).max() == 0:
                break
        else:
            continue
        c = rng.integers(3)
        if c == 0:
            shape = (abs(xx - cx) < s / 2) & (abs(yy - cy) < s / 2)
        elif c == 1:
            shape = (xx - cx) ** 2 + (yy - cy) ** 2 < (s / 2) ** 2
        else:
            top = cy - s / 2
            shape = (yy >= top) & (yy < cy + s / 2) & (abs(xx - cx) < (yy - top + 1) / 2)
        colour = rng.uniform(0, 1, 3)
        while np.abs(colour - bg).max() < 0.4:          # keep it visible
            colour = rng.uniform(0, 1, 3)
        img[shape] = colour + rng.normal(0, 0.08, (shape.sum(), 3))
        mask[shape] = c + 1
        ys, xs = np.nonzero(shape)                      # tight box around the pixels
        boxes.append([xs.min(), ys.min(), xs.max() + 1, ys.max() + 1])
        labels.append(c)
    return np.clip(img, 0, 1).astype(np.float32), np.array(boxes, float), np.array(labels), mask

def make_dataset(n, seed):
    rng = np.random.default_rng(seed)
    scenes = [make_scene(rng) for _ in range(n)]
    images = torch.tensor(np.stack([s[0] for s in scenes])).permute(0, 3, 1, 2)
    return images, [s[1] for s in scenes], [s[2] for s in scenes], [s[3] for s in scenes]

train_x, train_boxes, train_labels, train_masks = make_dataset(3000, seed=1)
test_x, test_boxes, test_labels, test_masks = make_dataset(500, seed=2)
print(train_x.shape, sum(len(b) for b in train_boxes), "objects")
print(train_boxes[0], [CLASSES[c] for c in train_labels[0]])

Always look at your data and labels before training:

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, i in zip(axes.flat, range(8)):
    ax.imshow(train_x[i].permute(1, 2, 0))
    for (x1, y1, x2, y2), c in zip(train_boxes[i], train_labels[i]):
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color="white", lw=1.5))
        ax.text(x1, y1 - 1, CLASSES[c], color="white", fontsize=8)
    ax.axis("off")
plt.show()

### A grid detector

Three max-pools shrink 64x64 to an 8x8 grid. Each cell predicts objectness, three class logits and a box. The cell holding an object's centre is responsible for it. This takes about a minute on a laptop CPU.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

STRIDE = 8                       # each output cell covers 8x8 pixels
GRID = SIZE // STRIDE            # 8x8 grid of cells

def encode_targets(boxes_list, labels_list):
    """Per cell: objectness, class, and box as (centre offset in cell, size)."""
    n = len(boxes_list)
    obj = torch.zeros(n, GRID, GRID)
    cls = torch.zeros(n, GRID, GRID, dtype=torch.long)
    box = torch.zeros(n, 4, GRID, GRID)
    for i, (bxs, lbs) in enumerate(zip(boxes_list, labels_list)):
        for (x1, y1, x2, y2), c in zip(bxs, lbs):
            cx, cy = (x1 + x2) / 2 / STRIDE, (y1 + y2) / 2 / STRIDE
            gx, gy = int(cx), int(cy)                  # the cell holding the centre
            obj[i, gy, gx] = 1
            cls[i, gy, gx] = c
            box[i, :, gy, gx] = torch.tensor([cx - gx, cy - gy,
                                              (x2 - x1) / SIZE, (y2 - y1) / SIZE])
    return obj, cls, box

def conv_block(c_in, c_out):
    return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1),
                         nn.BatchNorm2d(c_out), nn.ReLU())

class GridDetector(nn.Module):
    def __init__(self, n_classes=3):
        super().__init__()
        self.backbone = nn.Sequential(
            conv_block(3, 16), nn.MaxPool2d(2),        # 64 -> 32
            conv_block(16, 32), nn.MaxPool2d(2),       # 32 -> 16
            conv_block(32, 64), nn.MaxPool2d(2),       # 16 -> 8
            conv_block(64, 64), conv_block(64, 64))
        self.head = nn.Conv2d(64, 1 + n_classes + 4, 1)

    def forward(self, x):                              # x: (B, 3, 64, 64)
        out = self.head(self.backbone(x))              # (B, 8, 8, 8)
        return out[:, 0], out[:, 1:4], out[:, 4:].sigmoid()

def detection_loss(pred, target):
    obj_logit, cls_logit, box = pred
    obj, cls, box_t = target
    pos = obj > 0                                      # cells that hold an object
    loss_obj = F.binary_cross_entropy_with_logits(obj_logit, obj)
    loss_cls = F.cross_entropy(cls_logit.permute(0, 2, 3, 1)[pos], cls[pos])
    loss_box = F.l1_loss(box.permute(0, 2, 3, 1)[pos], box_t.permute(0, 2, 3, 1)[pos])
    return loss_obj + loss_cls + 5 * loss_box

torch.manual_seed(0)
targets = encode_targets(train_boxes, train_labels)
det = GridDetector()
epochs, batch = 8, 64
opt = torch.optim.AdamW(det.parameters(), lr=3e-3)
sched = torch.optim.lr_scheduler.OneCycleLR(
    opt, max_lr=3e-3, total_steps=epochs * -(-len(train_x) // batch))
for epoch in range(epochs):
    det.train()
    perm = torch.randperm(len(train_x))
    for i in range(0, len(perm), batch):
        idx = perm[i:i + batch]
        loss = detection_loss(det(train_x[idx]), [t[idx] for t in targets])
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    if epoch % 2 == 1:
        print(f"epoch {epoch} loss {loss.item():.3f}")

### Decoding and evaluating

Turn cells into boxes, score them, run class-aware NMS, and measure with `average_precision` from session 1.

In [ ]:
from torchvision.ops import batched_nms

def decode(pred, score_thresh=0.05, iou_thresh=0.5):
    """Turn grid outputs into a list of (boxes, scores, labels) per image."""
    obj_logit, cls_logit, box = pred
    gy, gx = torch.meshgrid(torch.arange(GRID), torch.arange(GRID), indexing="ij")
    cx = (gx + box[:, 0]) * STRIDE
    cy = (gy + box[:, 1]) * STRIDE
    w, h = box[:, 2] * SIZE, box[:, 3] * SIZE
    all_boxes = torch.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], -1)
    cls_prob, labels = cls_logit.softmax(1).max(1)
    scores = obj_logit.sigmoid() * cls_prob
    results = []
    for b, s, l in zip(all_boxes, scores, labels):
        b, s, l = b.reshape(-1, 4), s.reshape(-1), l.reshape(-1)
        keep = s > score_thresh
        b, s, l = b[keep], s[keep], l[keep]
        keep = batched_nms(b, s, l, iou_thresh)        # NMS within each class
        results.append((b[keep].numpy(), s[keep].numpy(), l[keep].numpy()))
    return results

def evaluate(results, gt_boxes, gt_labels, iou_thresh):
    aps = []
    for c in range(len(CLASSES)):
        d_img, d_score, d_box = [], [], []
        for i, (b, s, l) in enumerate(results):
            d_img += [i] * int((l == c).sum())
            d_score.append(s[l == c]); d_box.append(b[l == c])
        g = [bx[lb == c] for bx, lb in zip(gt_boxes, gt_labels)]
        aps.append(average_precision(np.array(d_img), np.concatenate(d_score),
                                     np.concatenate(d_box), g, iou_thresh))
    return aps

det.eval()
with torch.no_grad():
    results = decode(det(test_x))
for c, ap in zip(CLASSES, evaluate(results, test_boxes, test_labels, 0.5)):
    print(f"AP@0.5 {c:>8s}: {ap:.3f}")
maps = [np.mean(evaluate(results, test_boxes, test_labels, t)) for t in thresholds]
print(f"mAP@0.5 {maps[0]:.3f}   mAP@[.5:.95] {np.mean(maps):.3f}")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, i in zip(axes.flat, range(8)):
    ax.imshow(test_x[i].permute(1, 2, 0))
    b, s, l = results[i]
    for (x1, y1, x2, y2), score, c in zip(b, s, l):
        if score < 0.3:
            continue
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color="yellow", lw=1.5))
        ax.text(x1, y1 - 1, f"{CLASSES[c]} {score:.2f}", color="yellow", fontsize=7)
    ax.axis("off")
plt.show()

### Augmentation that moves the boxes

Whatever you do to the pixels, do to the boxes. A flip maps x1' = W - x2 and x2' = W - x1.

In [ ]:
from torchvision import tv_tensors
from torchvision.transforms import v2

def hflip(image, boxes):
    """image: (C, H, W) tensor; boxes: (N, 4) xyxy in pixels."""
    W = image.shape[-1]
    x1, y1, x2, y2 = boxes.unbind(-1)
    return image.flip(-1), torch.stack([W - x2, y1, W - x1, y2], -1)

img, bxs = train_x[0], torch.tensor(train_boxes[0])
img_f, bxs_f = hflip(img, bxs)
print(bxs)
print(bxs_f)

tv_boxes = tv_tensors.BoundingBoxes(bxs, format="XYXY", canvas_size=(SIZE, SIZE))
img_v2, bxs_v2 = v2.RandomHorizontalFlip(p=1.0)(tv_tensors.Image(img), tv_boxes)
print(torch.equal(img_v2, img_f), torch.allclose(bxs_v2, bxs_f))

### Hands-on: where does the detector go wrong?

A single mAP number hides what kind of mistakes a detector makes. Break them down.

1. Write `error_analysis(results, gt_boxes, gt_labels, score_thresh)`. Keep detections scoring at least `score_thresh` and label each one: *correct* (IoU of 0.5 or more with a same-class object not found yet), *duplicate* (IoU of 0.5 or more with an object already found), *localisation* (right class, IoU between 0.1 and 0.5), *wrong class* (IoU of 0.5 or more with an object of another class) or *background*. Also count the true objects that nothing found.
2. Run it on the test set at score thresholds 0.5, 0.3 and 0.1. How do missed objects and false positives trade off?
3. For the objects missed at 0.5, measure how far each centre lies from the nearest grid-cell border, in cell units, and compare with the objects that were found. What do you see, and why might the grid design cause it?

In [ ]:
# Your code here

## Segmentation

### Pixel-wise classification

Semantic segmentation gives every pixel a class; `F.cross_entropy` accepts logits of shape (batch, classes, height, width).

### A small U-Net

An encoder halves the resolution, a decoder upsamples back, and skip connections concatenate encoder features of the same size so edges stay sharp. Training takes about a minute on a laptop CPU.

In [ ]:
def double_conv(c_in, c_out):
    return nn.Sequential(conv_block(c_in, c_out), conv_block(c_out, c_out))

class UNet(nn.Module):
    def __init__(self, n_classes=4, c=8):
        super().__init__()
        self.enc1 = double_conv(3, c)                   # 64x64
        self.enc2 = double_conv(c, 2 * c)               # 32x32
        self.bottom = double_conv(2 * c, 4 * c)         # 16x16
        self.up2 = nn.ConvTranspose2d(4 * c, 2 * c, 2, stride=2)
        self.dec2 = double_conv(4 * c, 2 * c)           # input: upsampled + skip
        self.up1 = nn.ConvTranspose2d(2 * c, c, 2, stride=2)
        self.dec1 = double_conv(2 * c, c)
        self.out = nn.Conv2d(c, n_classes, 1)           # a class score per pixel

    def forward(self, x):
        s1 = self.enc1(x)
        s2 = self.enc2(F.max_pool2d(s1, 2))
        b = self.bottom(F.max_pool2d(s2, 2))
        d2 = self.dec2(torch.cat([self.up2(b), s2], dim=1))   # skip connection
        d1 = self.dec1(torch.cat([self.up1(d2), s1], dim=1))  # skip connection
        return self.out(d1)                             # (B, n_classes, 64, 64)

train_y = torch.tensor(np.stack(train_masks[:2000]))
test_y = torch.tensor(np.stack(test_masks))
torch.manual_seed(0)
unet = UNet()
opt = torch.optim.AdamW(unet.parameters(), lr=3e-3)
for epoch in range(4):
    unet.train()
    perm = torch.randperm(len(train_y))
    for i in range(0, len(perm), 32):
        idx = perm[i:i + 32]
        loss = F.cross_entropy(unet(train_x[idx]), train_y[idx])   # per pixel
        opt.zero_grad(); loss.backward(); opt.step()
    print(f"epoch {epoch} loss {loss.item():.3f}")

### Dice and IoU

Pixel accuracy is dominated by the background. Use per-class Dice or IoU.

In [ ]:
def seg_scores(pred, target, n_classes=4):
    """Per-class Dice and IoU, pooled over all pixels of all images."""
    dice, iou = [], []
    for c in range(1, n_classes):                       # skip background
        p, t = pred == c, target == c
        inter = (p & t).sum().item()
        total = p.sum().item() + t.sum().item()
        dice.append(2 * inter / total)
        iou.append(inter / (total - inter))
    return np.array(dice), np.array(iou)

unet.eval()
with torch.no_grad():
    pred = unet(test_x).argmax(1)                       # (500, 64, 64)
dice, iou = seg_scores(pred, test_y)
for c, d, j in zip(CLASSES, dice, iou):
    print(f"{c:>8s}  Dice {d:.3f}  IoU {j:.3f}")
print(f"pixel accuracy {(pred == test_y).float().mean():.3f}")
print(f"all-background baseline {(test_y == 0).float().mean():.3f}")

Where are the remaining errors? White pixels in the bottom row are wrong.

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(10, 7.5))
for col in range(4):
    axes[0, col].imshow(test_x[col].permute(1, 2, 0))
    axes[1, col].imshow(test_y[col], vmin=0, vmax=3, cmap="tab10")
    axes[2, col].imshow(pred[col] != test_y[col], cmap="gray")   # white = wrong pixel
for ax, name in zip(axes[:, 0], ["image", "true mask", "errors"]):
    ax.set_ylabel(name)
for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([])
plt.show()

### Transposed convolution or upsampling?

A 3x3 transposed convolution with stride 2 overlaps unevenly, the source of checkerboard artefacts. Kernel size equal to the stride, or resize followed by a convolution, avoids it.

In [ ]:
up = nn.ConvTranspose2d(1, 1, kernel_size=3, stride=2, bias=False)
nn.init.ones_(up.weight)
with torch.no_grad():
    print(up(torch.ones(1, 1, 4, 4))[0, 0, :6, :6])    # uneven overlap

up2 = nn.ConvTranspose2d(1, 1, kernel_size=2, stride=2, bias=False)
nn.init.ones_(up2.weight)
with torch.no_grad():
    print(up2(torch.ones(1, 1, 4, 4))[0, 0, :4, :4])   # no overlap: even

resize_conv = nn.Sequential(nn.Upsample(scale_factor=2, mode="bilinear"),
                            nn.Conv2d(16, 8, 3, padding=1))
print(resize_conv(torch.randn(1, 16, 16, 16)).shape)

## Pretrained detectors in practice

This cell downloads a 74 MB COCO-trained Faster R-CNN and runs it on a scene drawn with matplotlib. Nothing after it depends on it. A real photo gives a fairer test: `torchvision.io.decode_image("photo.jpg")`.

In [ ]:
# Downloads a model or data on first run
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, RegularPolygon, Rectangle
from torchvision.models.detection import (
    fasterrcnn_mobilenet_v3_large_fpn, FasterRCNN_MobileNet_V3_Large_FPN_Weights)

# draw a test scene: a stop sign and a wall clock (two COCO classes)
fig = plt.figure(figsize=(4, 3), dpi=100)
ax = fig.add_axes([0, 0, 1, 1], xlim=(0, 400), ylim=(0, 300))
ax.axis("off")
ax.add_patch(Rectangle((0, 0), 400, 300, color="#9db8cf"))
ax.add_patch(Rectangle((0, 0), 400, 70, color="#6b8e4e"))
ax.add_patch(Rectangle((112, 30), 10, 110, color="#888888"))
ax.add_patch(RegularPolygon((117, 180), 8, radius=62, orientation=np.pi / 8,
                            facecolor="#c8102e", edgecolor="white", lw=4))
ax.text(117, 180, "STOP", color="white", fontsize=22, weight="bold",
        ha="center", va="center")
ax.add_patch(Circle((300, 190), 55, facecolor="white", edgecolor="black", lw=5))
for k in range(12):
    a = k * np.pi / 6
    ax.plot([300 + 45 * np.cos(a), 300 + 52 * np.cos(a)],
            [190 + 45 * np.sin(a), 190 + 52 * np.sin(a)], color="black", lw=2)
ax.plot([300, 300], [190, 230], color="black", lw=4)
ax.plot([300, 330], [190, 175], color="black", lw=4)
fig.canvas.draw()
rgb = np.asarray(fig.canvas.buffer_rgba())[..., :3].copy()
plt.close(fig)
image = torch.from_numpy(rgb).permute(2, 0, 1)          # (3, H, W) uint8

weights = FasterRCNN_MobileNet_V3_Large_FPN_Weights.COCO_V1
model = fasterrcnn_mobilenet_v3_large_fpn(weights=weights).eval()
with torch.no_grad():
    out = model([weights.transforms()(image)])[0]
names = weights.meta["categories"]
for box, label, score in zip(out["boxes"], out["labels"], out["scores"]):
    if score > 0.5:
        print(f"{names[label]:>10s}  {score:.2f}  {box.round().int().tolist()}")

In [ ]:
# Downloads a model or data on first run
import matplotlib.pyplot as plt
from torchvision.utils import draw_bounding_boxes

keep = out["scores"] > 0.5
drawn = draw_bounding_boxes(image, out["boxes"][keep],
                            [names[i] for i in out["labels"][keep]], colors="yellow", width=3)
plt.imshow(drawn.permute(1, 2, 0)); plt.axis("off"); plt.show()

## Promptable and open-vocabulary models

### Segment anything

SAM returns a mask for whatever you point at with clicks or a box: a heavy image encoder runs once, then a light decoder answers each prompt quickly.

### Open-vocabulary detection

Replace the fixed classification layer with a comparison between region embeddings and text embeddings of your prompts (as in CLIP, lesson 23), so new classes are just new text. Try OWLv2 with Hugging Face `transformers`: `pipeline("zero-shot-object-detection", model="google/owlv2-base-patch16-ensemble")`.

## Video: tracking by detection

Run a detector per frame, then link detections across frames by IoU.

In [ ]:
def iou_track(frames, iou_thresh=0.3):
    """Greedy IoU tracker. frames: list of (N_t, 4) detection arrays.
    Returns one array of track IDs per frame."""
    tracks, next_id, all_ids = {}, 0, []          # tracks: id -> last box
    for dets in frames:
        ids = np.full(len(dets), -1)
        if tracks and len(dets):
            track_ids = np.array(list(tracks))
            iou = iou_matrix(np.array(list(tracks.values())), dets)
            pairs = np.unravel_index(np.argsort(-iou, axis=None), iou.shape)
            for t, d in zip(*pairs):              # best overlaps first
                if iou[t, d] < iou_thresh:
                    break
                if ids[d] == -1 and track_ids[t] not in ids:
                    ids[d] = track_ids[t]
        for d in np.flatnonzero(ids == -1):       # unmatched: start a new track
            ids[d] = next_id
            next_id += 1
        tracks = dict(zip(ids, dets))             # unmatched tracks end here
        all_ids.append(ids)
    return all_ids

# a synthetic video: three objects moving in straight lines, noisy boxes,
# returned in a random order each frame (as a detector would)
rng = np.random.default_rng(0)
start = np.array([[5, 10, 25, 30], [70, 5, 90, 25], [10, 70, 30, 90]], float)
velocity = np.array([[2, 1], [0, 2], [3, 0]], float)
frames, order = [], []
for t in range(20):
    boxes = start + np.tile(velocity * t, 2) + rng.normal(0, 1, (3, 4))
    perm = rng.permutation(3)
    frames.append(boxes[perm]); order.append(perm)

ids = iou_track(frames)
for obj in range(3):
    seen = {int(i[list(p).index(obj)]) for i, p in zip(ids, order)}
    print(f"object {obj} got track IDs {seen}")

In [ ]:
import matplotlib.pyplot as plt

for frame, frame_ids in zip(frames, ids):
    for box, i in zip(frame, frame_ids):
        cx, cy = (box[0] + box[2]) / 2, (box[1] + box[3]) / 2
        plt.scatter(cx, cy, color=f"C{i}", s=12)
plt.gca().invert_yaxis()                       # image coordinates: y grows downwards
plt.title("box centres coloured by track ID"); plt.show()

### Hands-on: keep tracks alive through missed detections

Real detectors miss objects now and then, and the basic tracker gives an object a new ID every time it reappears.

1. Drop each detection in `frames` with probability 0.2 (use `np.random.default_rng(3)`), and count how many distinct track IDs the basic tracker creates for the three objects.
2. Write `iou_track_aged(frames, iou_thresh=0.3, max_age=3)`, which keeps an unmatched track, with its last box, for up to `max_age` frames before deleting it.
3. Count the IDs again. You are done when the count is 3.

In [ ]:
# Your code here

## Depth and 3-D in brief

Monocular depth models (MiDaS, Depth Anything) mostly predict relative depth. With depth Z, focal length f and principal point (cx, cy), pixel (u, v) maps to X = (u - cx)Z/f, Y = (v - cy)Z/f.

## Deploying vision models

Resolution, a smaller architecture, export (ONNX, TensorRT, OpenVINO, Core ML, LiteRT) and int8 quantisation are the main levers. This cell uses random weights, so it needs no download; your timings will depend on your machine.

In [ ]:
import time
from torch.utils.flop_counter import FlopCounterMode
from torchvision.models import mobilenet_v3_large

backbone = mobilenet_v3_large(weights=None).features.eval()  # speed only
for side in (224, 320, 480, 640):
    x = torch.randn(1, 3, side, side)
    with torch.inference_mode():
        with FlopCounterMode(display=False) as counter:
            backbone(x)                                       # also a warm-up
        start_t = time.perf_counter()
        for _ in range(10):
            backbone(x)
    ms = (time.perf_counter() - start_t) / 10 * 1000
    gflops = counter.get_total_flops() / 1e9
    print(f"{side}x{side}: {gflops:5.2f} GFLOPs  {ms:5.1f} ms")

## Labelling and data quality

Write a labelling guide, measure annotator agreement with IoU, use a tool such as CVAT or Label Studio, label what the model finds hard (active learning), sort by loss to find label errors, and split train and test by source to avoid leakage (lesson 03).

## Check yourself

Take the quiz at the end of the lesson page. Then close everything and write three or four sentences in your learning log about what you learned, from memory.